[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Digital-AI-Finance/Introduction-to-Machine-Learning-notebooks/blob/master/trees.ipynb)

# Decision trees and random forests, in arithmetic you can check

Every number on the slides is computed here, and every answer is written
underneath the question, so you cannot get stuck.

If a cell fails, read the message and run the cells above it in order.

## 1. Eight rows, and the split the search picks

The eight rows of the worked example. Two measurements in whole millimetres,
and a label that is A or B.

In [ ]:
import numpy as np

X = np.array([[1, 1], [2, 1], [1, 2], [2, 3],
              [3, 3], [4, 2], [4, 1], [3, 1]], dtype=float)
y = np.array([0, 0, 0, 1, 1, 1, 0, 0])          # 0 is class A, 1 is class B
names = ["width", "height"]

print("rows and columns:", X.shape)
print("class A:", int((y == 0).sum()), "  class B:", int((y == 1).sum()))

The impurity of a box is one minus the sum of the squared class shares. Write
it once and use it on every box below.

In [ ]:
def gini(labels):
    if len(labels) == 0:
        return 0.0
    shares = np.bincount(labels, minlength=2) / len(labels)
    return float(1 - (shares ** 2).sum())

root = gini(y)
print("root impurity: %.4f" % root)
print("as a fraction: %d/64" % round(root * 64))

A threshold between two neighbouring values splits the rows the same way
wherever it sits, so the midpoints are the whole search. Score every one of
them.

In [ ]:
def gain(X, y, col, t):
    left, right = y[X[:, col] < t], y[X[:, col] >= t]
    n = len(y)
    return gini(y) - len(left) / n * gini(left) - len(right) / n * gini(right)

rows = []
for col in (1, 0):
    values = np.unique(X[:, col])
    for t in (values[:-1] + values[1:]) / 2:
        rows.append((names[col], t, gain(X, y, col, t)))

for name, t, g in rows:
    print("%-7s at %.1f   removes %.4f" % (name, t, g))
best = max(rows, key=lambda r: r[2])
print("\nwinner: %s at %.1f, removing %.3f" % best)

Five candidate splits, where two measurements and eight rows allow fourteen:
height takes three values and width four, so eleven of the fourteen thresholds
are repeats. The winner removes 0.281 and leaves four rows of class A alone.

scikit-learn searches the same five and picks the same one.

In [ ]:
from sklearn.tree import DecisionTreeClassifier

stump = DecisionTreeClassifier(max_depth=1).fit(X, y)
print("scikit-learn splits on %s at %.1f"
      % (names[stump.tree_.feature[0]], stump.tree_.threshold[0]))

## 2. The boxes the finished tree draws

Two splits, three leaves, and every leaf holds one class.

In [ ]:
import matplotlib.pyplot as plt

tree = DecisionTreeClassifier().fit(X, y)
gx, gy = np.meshgrid(np.linspace(0.4, 4.6, 300), np.linspace(0.4, 3.6, 300))
zone = tree.predict(np.c_[gx.ravel(), gy.ravel()]).reshape(gx.shape)

fig, ax = plt.subplots(figsize=(5, 3))
ax.contourf(gx, gy, zone, levels=[-0.5, 0.5, 1.5],
            colors=["#dfe4ec", "white"])
ax.plot(X[y == 0, 0], X[y == 0, 1], "o", color="#1e3a5f", ms=7, label="class A")
ax.plot(X[y == 1, 0], X[y == 1, 1], "s", mfc="none", mec="#1e3a5f", mew=1.5,
        ms=7, label="class B")
ax.set_xlabel("width")
ax.set_ylabel("height")
ax.set_title("leaves: %d,  right on %d of %d rows"
             % (tree.get_n_leaves(), (tree.predict(X) == y).sum(), len(y)))
ax.legend(loc="upper left", frameon=False)
plt.show()

## 3. Where the error rate stops telling two splits apart

Eight hundred rows, four hundred of each class, and two ways to split them.
Both leave two hundred rows on the wrong side.

In [ ]:
def counts_gini(c):
    n = sum(c)
    return 1 - sum((k / n) ** 2 for k in c)

def counts_entropy(c):
    n = sum(c)
    return -sum((k / n) * np.log2(k / n) for k in c if k)

def counts_error(c):
    return 1 - max(c) / sum(c)

parent = (400, 400)
splits = {"first": ((300, 100), (100, 300)), "second": ((200, 400), (200, 0))}

for label, measure in (("error rate", counts_error), ("Gini", counts_gini),
                       ("entropy", counts_entropy)):
    line = []
    for name, children in splits.items():
        n = sum(parent)
        removed = measure(parent) - sum(sum(c) / n * measure(c)
                                        for c in children)
        line.append("%s %.3f" % (name, removed))
    print("%-11s %s" % (label, "   ".join(line)))

The error rate removes 0.250 either way. Gini removes 0.125 and 0.167, entropy
0.189 and 0.311: both prefer the split that empties a box, and the error rate
has no way to say so. That is why growth is run on Gini or entropy.

## 4. A real table, and what an unpruned tree does on it

scikit-learn ships the breast cancer table: 569 rows, 30 measurements, two
classes. Nothing is downloaded.

In [ ]:
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split

data = load_breast_cancer()
Xtr, Xte, ytr, yte = train_test_split(data.data, data.target, test_size=0.3,
                                      random_state=0)
print("learning from:", Xtr.shape[0], "rows   held back:", Xte.shape[0])

depths = range(1, 13)
train = [DecisionTreeClassifier(max_depth=d, random_state=0)
         .fit(Xtr, ytr).score(Xtr, ytr) for d in depths]
held = [DecisionTreeClassifier(max_depth=d, random_state=0)
        .fit(Xtr, ytr).score(Xte, yte) for d in depths]

fig, ax = plt.subplots(figsize=(5.4, 2.8))
ax.plot(list(depths), train, "--", color="#64748b", label="training rows")
ax.plot(list(depths), held, color="#1e3a5f", label="rows held back")
ax.set_xlabel("depth allowed")
ax.set_ylabel("share right")
ax.legend(frameon=False)
plt.show()

print("deepest tree: %.3f on the training rows, %.3f on the rows held back"
      % (train[-1], held[-1]))

The training curve reaches 1.000 and stays there. The held back curve stops
improving around depth four and then wanders: everything after that is the
tree fitting rows it will never see again.

## 5. Averaging trees, and the identity that says what it buys

The variance of an average of B trees, each of variance sigma squared and each
pair correlated rho, is

    rho * sigma^2 + (1 - rho) * sigma^2 / B

Fit one forest, then draw subsets of its trees and measure the variance of
their average against that formula.

In [ ]:
from sklearn.ensemble import RandomForestClassifier

forest = RandomForestClassifier(n_estimators=200, random_state=0).fit(Xtr, ytr)
P = np.array([t.predict_proba(Xte)[:, 1] for t in forest.estimators_])

sigma2 = P.var(axis=0).mean()                      # one tree, averaged over rows
C = np.corrcoef(P[P.std(axis=1) > 0])
rho = float(np.nanmean(C[~np.eye(C.shape[0], dtype=bool)]))
print("one tree: sigma^2 = %.4f    correlation between two trees: %.3f"
      % (sigma2, rho))

rng = np.random.default_rng(0)
sizes = [1, 2, 5, 10, 25, 50, 100]
measured = []
for B in sizes:
    means = [P[rng.choice(len(P), B, replace=False)].mean(axis=0)
             for _ in range(60)]
    measured.append(np.var(np.array(means), axis=0).mean())

formula = [rho * sigma2 + (1 - rho) * sigma2 / B for B in sizes]

fig, ax = plt.subplots(figsize=(5.4, 2.8))
ax.plot(sizes, measured, "o", color="#1e3a5f", label="measured")
ax.plot(sizes, formula, color="#b45309", label="the identity")
ax.set_xlabel("trees averaged")
ax.set_ylabel("variance of the average")
ax.set_ylim(0)
ax.legend(frameon=False)
plt.show()

print("floor the identity leaves: %.4f" % (rho * sigma2))

The dots sit on the line, and both flatten out at rho times sigma squared.
More trees buy the second term and nothing else, which is why a forest of a
thousand trees scores about what a forest of two hundred scores.

## 6. Lowering the correlation: m of the p measurements at each split

The floor moves only if the trees stop agreeing. Drawing a few measurements at
each split is what moves it. Sixty columns, five of them carrying the signal.

In [ ]:
from sklearn.datasets import make_classification

Xm, ym = make_classification(n_samples=900, n_features=60, n_informative=5,
                             n_redundant=0, n_repeated=0, class_sep=0.9,
                             flip_y=0.01, random_state=0)
Xmtr, Xmte, ymtr, ymte = train_test_split(Xm, ym, test_size=0.3,
                                          random_state=0)

def spread(fit, Xs):
    Q = np.array([t.predict_proba(Xs)[:, 1] for t in fit.estimators_])
    G = np.corrcoef(Q[Q.std(axis=1) > 0])
    rho = float(np.nanmean(G[~np.eye(G.shape[0], dtype=bool)]))
    return rho, float(Q.var(axis=0).mean())

ms = [1, 3, 7, 15, 30, 60]
rhos, wrong = [], []
for m in ms:
    fit = RandomForestClassifier(n_estimators=60, max_features=m,
                                 random_state=0).fit(Xmtr, ymtr)
    rho_m, sigma2_m = spread(fit, Xmte)
    rhos.append(rho_m)
    wrong.append(1 - fit.score(Xmte, ymte))
    print("m = %-3d correlation %.3f   one tree %.4f   floor %.4f   "
          "got wrong %.3f"
          % (m, rho_m, sigma2_m, rho_m * sigma2_m, wrong[-1]))

The correlation climbs with m and the error falls and then flattens. The
default is the square root of the number of columns, seven of the sixty here,
and it is a rule of thumb: on this table the error is lowest further along.

## 7. The rows a bootstrap sample leaves out

Each tree is grown on n rows drawn with replacement, so a given row is left out
with probability (1 - 1/n) to the power n, which is 0.368 for any n worth
having.

In [ ]:
n = Xtr.shape[0]
print("rows: %d   share left out: %.4f   one over e: %.4f"
      % (n, (1 - 1 / n) ** n, np.exp(-1)))

scored = RandomForestClassifier(n_estimators=300, oob_score=True,
                                random_state=0).fit(Xtr, ytr)
print("scored on the rows each tree never saw: %.3f" % scored.oob_score_)
print("scored on the rows held back at the start: %.3f"
      % scored.score(Xte, yte))

Two numbers within a point of each other, and the first one cost no rows at
all: every row of the table was used for growing, and every row was also used
for scoring by the trees that never saw it.

## 8. Your turn

Two changes, and one prediction before you run anything.

**First.** Section 6 fits a forest at each `m` in `ms = [1, 3, 7, 15, 30, 60]`
and prints the correlation and the floor at each one. Say how far apart the
floors at `m = 1` and `m = 60` are, and which forest gets more wrong, then
change `n_estimators=60` to `n_estimators=200` and run the cell again.

**Second.** In section 1, replace `gini` with an entropy impurity and score the
five candidate splits again. Does the winner change?

What to expect. With sixty trees the floor at `m = 1` is 0.006 and the floor
at `m = 60` is 0.062, a factor of ten, and the forest with the lower floor is
the one that gets more wrong: its trees are too weak on their own for the
floor to be the whole story, which is the trade `m` sets. Going to two hundred
trees takes the error at `m = 1` from 0.281 to 0.204 and the error at `m = 60`
from 0.130 to 0.115: where the correlation is near zero, the term that falls
with `B` is still most of the spread and more trees keep paying. The entropy
ranking is the Gini ranking, height at 1.5 first and width at 3.5 last, so the
winner stands on these eight rows.